# 04 — Similitud entre Documentos
**Goal:** Medir qué tan parecidos son dos documentos usando álgebra lineal.

## Similitud coseno

$$\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|}$$

- Rango: `[0, 1]` con vectores TF-IDF (no negativos)
- `1` = idénticos, `0` = sin palabras en común
- **No depende de la longitud del documento** — ventaja clave sobre la distancia euclídea

Aplicaciones: deduplicar tickets, encontrar comentarios similares, búsqueda semántica básica.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
})

STOPWORDS_ES = [
    'a','al','algo','ante','como','con','de','del','desde','el','ella','en',
    'entre','es','esta','este','esto','fue','la','las','le','lo','los','me',
    'mi','muy','no','nos','o','para','pero','por','que','se','si','sin',
    'su','sus','también','te','todo','un','una','y','ya','yo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

## 1. Similitud coseno a mano

In [ ]:
# Dos vectores TF-IDF de ejemplo
a = np.array([0.8, 0.0, 0.6, 0.0, 0.0])  # doc con 'app' y 'lenta'
b = np.array([0.7, 0.0, 0.5, 0.0, 0.5])  # doc con 'app', 'lenta' y 'falla'
c = np.array([0.0, 0.9, 0.0, 0.8, 0.0])  # doc completamente diferente

def cosine(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

print(f'sim(a, b) = {cosine(a, b):.4f}  ← parecidos')
print(f'sim(a, c) = {cosine(a, c):.4f}  ← distintos')
print(f'sim(a, a) = {cosine(a, a):.4f}  ← idéntico')

## 2. Matriz de similitud entre todos los documentos

In [ ]:
# 10 tickets de soporte con temas distintos
tickets = [
    'La app está muy lenta y tarda en cargar',
    'La aplicación carga lento, no puedo entrar',
    'No me llega el OTP al celular',
    'El código OTP no llega por SMS',
    'Quiero cancelar mi tarjeta de crédito',
    'Necesito cancelar y dar de baja la tarjeta',
    'El proceso de solicitud fue muy rápido',
    'Excelente experiencia, aprobaron en minutos',
    'La carga de documentos falla constantemente',
    'No puedo subir los documentos requeridos',
]

tfidf = TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES)
X = tfidf.fit_transform(tickets)   # (10, vocab)

# Matriz de similitud (10, 10)
sim_matrix = cosine_similarity(X)  # sklearn hace el cálculo vectorizado

fig, ax = plt.subplots(figsize=(8, 6))
labels = [f'T{i}' for i in range(len(tickets))]
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(tickets)))
ax.set_yticks(range(len(tickets)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
ax.set_title('Matriz de similitud coseno entre tickets')
for i in range(len(tickets)):
    for j in range(len(tickets)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                fontsize=7, color='white' if sim_matrix[i,j] > 0.5 else 'black')
plt.tight_layout()
plt.show()

In [ ]:
# Imprimir pares más similares (excluyendo diagonal)
np.fill_diagonal(sim_matrix, 0)  # ignorar auto-similitud

print('Pares de tickets más similares:')
for _ in range(5):
    i, j = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
    print(f'\n  sim={sim_matrix[i,j]:.3f}')
    print(f'  T{i}: {tickets[i]}')
    print(f'  T{j}: {tickets[j]}')
    sim_matrix[i, j] = 0
    sim_matrix[j, i] = 0

## 3. Búsqueda por similitud — motor de recuperación básico

In [ ]:
# Dataset más grande de tickets
rng = np.random.default_rng(42)
ticket_templates = [
    'La app está muy lenta y tarda en cargar',
    'No me llega el OTP al celular',
    'Quiero cancelar mi tarjeta de crédito',
    'El proceso de solicitud fue muy rápido',
    'La carga de documentos falla constantemente',
    'No puedo ver mi estado de cuenta',
    'El call center no responde mis llamadas',
    'Quiero aumentar mi límite de crédito',
    'La evaluación crediticia lleva días sin respuesta',
    'Excelente servicio, recomiendo la tarjeta',
]

corpus = [ticket_templates[rng.integers(10)] for _ in range(200)]

# Fit vectorizador sobre el corpus
tfidf = TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES, ngram_range=(1,2))
X_corpus = tfidf.fit_transform(corpus)

def search(query, top_k=5):
    q_vec = tfidf.transform([query])  # vectorizar la query
    sims  = cosine_similarity(q_vec, X_corpus).ravel()  # (200,)
    top   = np.argsort(sims)[::-1][:top_k]
    print(f'Query: "{query}"\n')
    for rank, idx in enumerate(top, 1):
        print(f'  #{rank} (sim={sims[idx]:.3f}): {corpus[idx]}')

search('no funciona la aplicación')
print()
search('quiero subir de límite')

## 4. Deduplicación de tickets

In [ ]:
# Encontrar tickets duplicados (sim > umbral)
THRESHOLD = 0.95

sample_tickets = corpus[:50]
X_sample = tfidf.transform(sample_tickets)
sim = cosine_similarity(X_sample)
np.fill_diagonal(sim, 0)

duplicates = np.argwhere(sim > THRESHOLD)
# Mantener solo pares únicos (i < j)
duplicates = [(i, j) for i, j in duplicates if i < j]

print(f'Pares con similitud > {THRESHOLD}: {len(duplicates)}')
for i, j in duplicates[:5]:
    print(f'  ({i}, {j}) sim={sim[i,j]:.3f}')
    print(f'    A: {sample_tickets[i]}')
    print(f'    B: {sample_tickets[j]}')

## 5. Visualización — proyección 2D con SVD

In [ ]:
from sklearn.decomposition import TruncatedSVD

# Reducir a 2D para visualizar clusters temáticos
svd = TruncatedSVD(n_components=2, random_state=42)
X2d = svd.fit_transform(X_corpus)

# Colorear por tema (template de origen — sabemos el índice)
# Simplificación: usar los primeros 100 docs con etiqueta conocida
temas = {
    0: ('App lenta', '#e63946'),
    1: ('OTP', '#457b9d'),
    2: ('Cancelar tarjeta', '#2a9d8f'),
    3: ('Proceso rápido', '#06d6a0'),
    4: ('Docs', '#f4a261'),
}

fig, ax = plt.subplots(figsize=(10, 7))

# Graficar primeros 100 docs con color por template
for i, text in enumerate(corpus[:100]):
    # Inferir tema por coincidencia de substring
    if 'lenta' in text or 'carga' in text.split()[0:3]:
        color, label = '#e63946', 'App'
    elif 'otp' in text.lower():
        color, label = '#457b9d', 'OTP'
    elif 'cancel' in text.lower():
        color, label = '#2a9d8f', 'Cancelar'
    elif 'rápido' in text or 'excelente' in text:
        color, label = '#06d6a0', 'Positivo'
    else:
        color, label = '#aaa', 'Otro'
    ax.scatter(X2d[i, 0], X2d[i, 1], color=color, alpha=0.6, s=40)

ax.set_title('Tickets proyectados en 2D (TF-IDF + SVD)')
ax.set_xlabel('SVD dim 1')
ax.set_ylabel('SVD dim 2')
plt.tight_layout()
plt.show()

## Resumen
| Concepto | API |
|---|---|
| Similitud coseno | `cosine_similarity(X)` o `linear_kernel(X)` |
| Query search | `cosine_similarity(q_vec, X_corpus).ravel()` → argsort |
| Deduplicación | `np.argwhere(sim > threshold)` |
| Visualización | `TruncatedSVD(n_components=2)` → scatter |

**Siguiente:** `05_sentiment.ipynb` — detectar polaridad de comentarios con léxico y con ML.